L1正则化的思想是在损失函数中额外加入模型所有参数的绝对值之和作为惩罚项。这样模型在训练过程中，会尽可能的让参数变小来降低Loss值，直到让某些参数为0。如果一个参数降为0，则等价于这个参数消失，起到了降低模型复杂度的作用。同样减小参数的值也是降低模型复杂度的一种方式。L1正则化的公式为：


Loss 
L1
​
 =Loss(θ)+λ∑ 
i
​
 ∣θ 
i
​
 ∣

其中
θ
θ是模型参数，
λ
λ是控制正则化强度的参数，一般初始设置为1e-3或者1e-4。设置过大会导致模型欠拟合。在训练过程中可以进根据实际情况进行调整，调整原则为如果数据量越少，模型越复杂，输入特征越多，那么
λ
λ就越大。因为这些情况都更容易出现过拟合。反之则可以减小
λ
λ。

L2正则化与L1正则化非常类似，它在损失函数中加入模型参数的平方和作为惩罚项。同样可以达到控制模型复杂度的效果。它的公式为：


Loss 
L2
​
 =Loss(θ)+λ∑ 
i
​
 θ 
i
​
  
2
 

其中
λ
λ的初始化和配置与L1类似。

L1正则化会让模型很多参数为0，也就是模型参数会变得稀疏。L2正则化会让模型参数变小，但是不会等于0。为什么会产生这种效果呢？

因为L1正则化每个参数是以绝对值的形式加入Loss函数，绝对值函数的导数在x大于0为1，x小于0为-1，在0处没有定义。所以它可以按照固定速度降低参数的绝对值，直到为0。L2正则化每个参数是以平方项的形式加入Loss函数，平方项的导数为
2
x
2x，当
x
x越接近零，梯度会越小，所以参数只会接近0，而不会等于0。

当然调整参数过程大部分还是受原本Loss函数的影响。它和正则项一起优化出一个既有效又简单的模型来。L2正则化让所有参数均匀缩小，避免模型过度依赖某些特征，同时保留所有特征的贡献，适用于大多数场景。所以一般情况下都默认使用L2正则化，很少使用L1正则化。

In [ ]:
l2_norm = 0.0
for param in model.parameters():
      l2_norm += param.pow(2).sum()
loss = criterion(outputs, labels) + 1e-4 * l2_norm

在PyTorch里增加L2正则化非常简单，就是遍历所有参数，对所有参数进行平方并加和。最终将这个值乘以一个很小的系数，比如上边代码里的1-e4，最终加入到loss中就可以。



指数加权

假如你新开了一家小店，并记录小店前六天的收入
接下来你想要预测第七天的收入。那该用什么方法来预测呢？
一个简单的预测办法，就是用前六天的平均值来预测。也就是前边6天，在预测第7天收入时，所占的权重都是相等的，均为1/6。

第一天0.05权重，0.1，0.15....最后一天0.25


根据我们的经验，越新的数据越有参考价值。比如预测第七天收入时，第六天就应该比第一天更有参考价值，因为第七天的运营情况更类似于第六天，而不是第一天。所以给予离当前越近的数据越高的权重。

指数加权平均就利用了加权平均的思想，并且老的数据的权重是按照指数衰减的。

每天指数加权平均的值我们用
V
i
V 
i
​
 表示，每天实际的收入值用
θ
i
θ 
i
​
 表示。比如对于第四天的收入而言，
θ
4
θ 
4
​
 就是真实值，117。
V
3
V 
3
​
 就是利用前三天的收入值进行指数加权平均得到对第四天收入的预测。下边我们就来看一下如何用指数加权平均来预测你每天的收入。

首先我们设置初始条件，
V
0
=
0
,
β
=
0.7
V 
0
​
 =0,β=0.7。其中
β
β是一个超参数，它越小，越老的数据，权重衰减的越厉害，在最终预测结果里占的比重越小。 首先对第一天收入进行指数加权平均：

V
1
=
β
V
0
+
(
1
−
β
)
θ
1
V 
1
​
 =βV 
0
​
 +(1−β)θ 
1
​
 

前二天收入的指数加权平均值它需要用到第一天的指数加权平均结果
V
1
V 
1
​
 ，和第二天的收入值
θ
2
θ 
2
​
 ：

V
2
=
β
V
1
+
(
1
−
β
)
θ
2
V 
2
​
 =βV 
1
​
 +(1−β)θ 
2
​
 

前三天收入的指数加权平均值：

V
3
=
β
V
2
+
(
1
−
β
)
θ
3
V 
3
​
 =βV 
2
​
 +(1−β)θ 
3
​
 

我们可以用前三天的指数加权平均值来预测第四天的收入。

前
t
t步的指数加权平均值为：

V
t
=
β
V
i
−
t
+
(
1
−
β
)
θ
t
V 
t
​
 =βV 
i−t
​
 +(1−β)θ 
t
​
 

那么，对应到我们预测第七天收入的问题里：

V
0
=
0
,
β
=
0.7
V 
0
​
 =0,β=0.7

V
1
=
0.7
V
0
+
0.3
θ
1
V 
1
​
 =0.7V 
0
​
 +0.3θ 
1
​
 

V
2
=
0.7
V
1
+
0.3
θ
2
V 
2
​
 =0.7V 
1
​
 +0.3θ 
2
​
 

V
3
=
0.7
V
2
+
0.3
θ
3
V 
3
​
 =0.7V 
2
​
 +0.3θ 
3
​
 

V
4
=
0.7
V
3
+
0.3
θ
4
V 
4
​
 =0.7V 
3
​
 +0.3θ 
4
​
 

V
5
=
0.7
V
4
+
0.3
θ
5
V 
5
​
 =0.7V 
4
​
 +0.3θ 
5
​
 

V
6
=
0.7
V
5
+
0.3
θ
6
V 
6
​
 =0.7V 
5
​
 +0.3θ 
6
​
 

我们带入实际的值来进行计算前六天的指数加权平均值：

V
6
=
0.7
(
0.7
V
4
+
0.3
θ
5
)
+
0.3
θ
6
V 
6
​
 =0.7(0.7V 
4
​
 +0.3θ 
5
​
 )+0.3θ 
6
​
 

=
0.
7
2
V
4
+
0.3
×
0.7
θ
5
+
0.3
θ
6
=0.7 
2
 V 
4
​
 +0.3×0.7θ 
5
​
 +0.3θ 
6
​
 

=
0.
7
2
(
0.7
V
3
+
0.3
θ
4
)
+
0.3
×
0.7
θ
5
+
0.3
θ
6
=0.7 
2
 (0.7V 
3
​
 +0.3θ 
4
​
 )+0.3×0.7θ 
5
​
 +0.3θ 
6
​
 

=
0.
7
3
V
3
+
0.3
×
0.
7
2
θ
4
+
0.3
×
0.7
θ
5
+
0.3
θ
6
=0.7 
3
 V 
3
​
 +0.3×0.7 
2
 θ 
4
​
 +0.3×0.7θ 
5
​
 +0.3θ 
6
​
 

继续带入V值，最终得到：

V
6
=
0.3
×
0.
7
5
θ
1
+
0.3
×
0.
7
4
θ
2
+
0.3
×
0.
7
3
θ
3
+
0.3
×
0.
7
2
θ
4
+
0.3
×
0.7
θ
5
+
0.3
θ
6
V 
6
​
 =0.3×0.7 
5
 θ 
1
​
 +0.3×0.7 
4
 θ 
2
​
 +0.3×0.7 
3
 θ 
3
​
 +0.3×0.7 
2
 θ 
4
​
 +0.3×0.7θ 
5
​
 +0.3θ 
6
​
 

观察上式，你可以看到越老数据的系数
β
β的指数越大，因为
β
β小于1，所以指数越大，权重越小。

这样我们就可以用前
t
t天的指数加权平均值来作为对
t
+
1
t+1天收入的预测。比如用前6天的指数加权平均值来作为对第7天收入的预测。

在实际中，我们并不需要定义
V
0
,
V
1
,
.
.
.
,
V
6
V 
0
​
 ,V 
1
​
 ,...,V 
6
​
 ,我们只需要定义一个变量
V
V,并在指数加权平均计算过程中不断更新
V
V值就可以，比如：

V
=
0
,
β
=
0.7
V=0,β=0.7

V
=
0.7
V
+
0.3
θ
1
V=0.7V+0.3θ 
1
​
 

V
=
0.7
V
+
0.3
θ
2
V=0.7V+0.3θ 
2
​
 

...

指数加权平均算法的优点是：1.算法简单。2.计算过程中只要保存一个变量V，节省存储。

通过对上边表格的观察，我们发现对于前几天的指数加权平均，得到的值非常小，和实际值偏差较大，这是由于初始设置
V
0
=
0
V 
0
​
 =0导致的。如果你计算的序列很长，越到后边的指数加权平均计算，
V
0
V 
0
​
 的影响就会越小，结果自然得到修正。如果你计算的序列短的话，有什么办法解决吗？ 办法是对
V
V进行修正，在我们计算完
V
V后，让它除以
1
−
β
t
1−β 
t
 。
t
t表示第几天。

V
0
=
0
V 
0
​
 =0

V
t
=
β
V
t
−
1
+
(
1
−
β
)
θ
t
V 
t
​
 =βV 
t−1
​
 +(1−β)θ 
t
​
 

V
t
c
o
r
r
e
c
t
=
V
t
1
−
β
t
V 
t
correct
​
= V 
t/
1−β^t
 

​
 
​


可以看到经过修正后的
V
V值就更准确了。我们再观察一下
1
−
β
t
1−β 
t
 。发现它逐渐接近于1，也就是说当序列足够长，后面对
V
V进行修正时基本等于除以1，也就是修正效果很小。